Steps involved

1. Define State
2. Create LLM
3. Create Chat Node
4. Build Graph(node+edge)
5. Compile
6. Test single Run
7. Add loop for real chat 

In [7]:
from dotenv import load_dotenv
load_dotenv()

import os

In [8]:
hf_token=os.getenv("HF_TOKEN")

In [19]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

# Define State

In [20]:
class State(TypedDict):
    messages:Annotated[list,add_messages]
    

# Create LLM


In [21]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
    huggingfacehub_api_token=hf_token,
    max_new_tokens=200,
    temperature=0.7
)

llm = ChatHuggingFace(llm=llm)

# Chat node

In [26]:
def chatbot(state:State):
    #convert LangGraph messages-> plain prompt
    prompt=""
    for msg in state["messages"]:
        role="User" if isinstance(msg,HumanMessage) else "Assistant"
        prompt+=f"{role}: {msg.content}\n"

    prompt+="Assistant"

    response=llm.invoke(prompt)
    return {"messages":[response]}

# Build and Compile Graph

In [27]:
graph=StateGraph(State)

graph.add_node("chatbot",chatbot)


graph.add_edge(START,"chatbot")
graph.add_edge("chatbot",END)

app=graph.compile()

# Test single run 

In [28]:
#initial empty state

test_state={
    "messages":[HumanMessage(content="Say hello in one sentence")]
}

#run graph once
result=app.invoke(test_state)

#print response
print("Bot", result["messages"][-1].content)
    

Bot Hello there!


# Add loop for real chat

In [ ]:
state={"messages":[]}

print("Chat bot is ready! Type 'exit' to stop \n")

while True:
    user_input=input("You: ")

    if(user_input.lower())=="exit":
        print("Bye Bye ")
        break

    state["messages"].append(HumanMessage(content=user_input))

    state=app.invoke(state)

    print("Bot:",state["messages"][-1].content)

    

Chat bot is ready! Type 'exit' to stop 



You:  Hi


Bot: Hello! How can I assist you today?


You:  which is greater 2 or 6


Bot: The number 6 is greater than the number 2.


You:  multiply smaller one with 5


Bot: Sure! The smaller number between 2 and 6 is 2. When you multiply 2 by 5, you get:

\[ 2 \times 5 = 10 \]

So, the result is 10.
